In [16]:
!pip install textstat textblob spacy tqdm
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [17]:
# -------------------------------------------------------------------
#  Import packages
# -------------------------------------------------------------------
import re
import string
import numpy as np
import pandas as pd

from tqdm import tqdm
from textblob import TextBlob
import textstat
import spacy

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

# Makes pandas progress_apply available
tqdm.pandas()

In [18]:
# -------------------------------------------------------------------
# Load train and test datasets (local)
# -------------------------------------------------------------------

train_df = pd.read_csv("Data/train.csv", sep=None, engine="python")
test_df = pd.read_csv("Data/test.csv", sep=None, engine="python")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (7000, 5)
Test shape: (1500, 5)


In [19]:
# -------------------------------------------------------------------
# Helper functions for stylometric feature extraction
# -------------------------------------------------------------------
# This part transforms each raw review into numerical features.
# The features describe writing style, for example length, punctuation,
# lexical diversity, readability, sentiment, POS usage, and repetition.
# -------------------------------------------------------------------


def safe_divide(a, b):
    """
    Divides a by b, but returns 0 if b is 0.

    This prevents errors for very short or empty reviews.
    For example, if word_count is 0, ratios like
    unique_word_count / word_count would normally fail.
    """
    return a / b if b != 0 else 0


def count_syllables_in_text(text):
    """
    Counts syllables in the review using textstat.

    Syllable count is needed for readability scores and gives an
    additional indication of word complexity.
    """
    try:
        return textstat.syllable_count(text)
    except:
        return 0


def extract_stylometric_features(text):
    """
    Extracts stylometric and linguistic features from one review.

    Input:
        text: one review as a string

    Output:
        features: a dictionary with numerical feature values

    Later, this dictionary becomes one row in the feature dataset.
    Each feature becomes one column, and the label tells the model
    whether the review is human-written or AI-generated.
    """
  # ---------------------------------------------------------------
    # Basic text preparation
    # ---------------------------------------------------------------
    # Make sure the input is a string. If the review is missing,
    # replace it with an empty string so that the code does not stop.
    if not isinstance(text, str):
        text = ""

    # Remove unnecessary spaces at the beginning and end.
    text = text.strip()

    # ---------------------------------------------------------------
    # Process the text with spaCy
    # ---------------------------------------------------------------
    # spaCy splits the text into tokens and sentences and assigns
    # linguistic information such as part-of-speech tags.
    # This is needed for POS-based features like noun_ratio,
    # verb_ratio, adjective_ratio, and pronoun_ratio.
    doc = nlp(text)

    # Keep all non-space tokens.
    tokens = [token for token in doc if not token.is_space]

    # Keep only alphabetic words.
    # This removes punctuation and numbers from the word-based counts.
    words = [token for token in tokens if token.is_alpha]

    # Lowercase version of all words, used for unique words and word lists.
    word_texts = [token.text.lower() for token in words]

    # Extract sentence objects from spaCy.
    sentences = list(doc.sents)

    # Count how many alphabetic words each sentence contains.
    # This is used for average sentence length and sentence length variation.
    sentence_lengths = [
        len([token for token in sent if token.is_alpha])
        for sent in sentences
    ]


    # Extract stylometric features

    # ---------------------------------------------------------------
    # 1. Length and structure features
    # ---------------------------------------------------------------
    # These features describe the general size and structure of a review.
    # They are useful because AI-generated reviews may be more regular,
    # longer, or more polished than human-written reviews.

    char_count = len(text)
    word_count = len(words)
    unique_word_count = len(set(word_texts))
    sentence_count = len(sentences)

    # Word lengths are used to calculate average word length and variation.
    word_lengths = [len(token.text) for token in words]

    # ---------------------------------------------------------------
    # 2. Punctuation features
    # ---------------------------------------------------------------
    # Punctuation is part of writing style. Human reviews may use more
    # informal punctuation, while AI reviews may use more standardized
    # punctuation patterns.

    punctuation_chars = [char for char in text if char in string.punctuation]
    punctuation_count = len(punctuation_chars)

    comma_count = text.count(",")
    period_count = text.count(".")
    exclamation_count = text.count("!")
    question_count = text.count("?")
    semicolon_count = text.count(";")
    colon_count = text.count(":")

    repeated_punctuation_count = len(re.findall(r"([!?.,])\1+", text))

    # ---------------------------------------------------------------
    # 3. Part-of-speech features
    # ---------------------------------------------------------------
    # POS ratios describe grammatical style.
    # For example, AI-generated reviews may use many adjectives or adverbs,
    # while human reviews may use more pronouns when describing personal
    # experience.

    noun_count = sum(1 for token in words if token.pos_ in ["NOUN", "PROPN"])
    verb_count = sum(1 for token in words if token.pos_ == "VERB")
    adj_count = sum(1 for token in words if token.pos_ == "ADJ")
    adv_count = sum(1 for token in words if token.pos_ == "ADV")
    pronoun_count = sum(1 for token in words if token.pos_ == "PRON")
    conjunction_count = sum(1 for token in words if token.pos_ in ["CCONJ", "SCONJ"])

    # ---------------------------------------------------------------
    # 4. Function-word and style-marker features
    # ---------------------------------------------------------------
    # Stopwords are common words such as "the", "and", "is".
    # They are often used in stylometry because they reflect writing habits
    # rather than the topic of the review.
    stopword_count = sum(1 for token in words if token.is_stop)

    # Negation words may be useful because negative reviews often contain
    # words such as "not", "never", or "don't".
    negation_words = {
        "no", "not", "never", "none", "nobody", "nothing", "neither",
        "nowhere", "cannot", "can't", "don't", "doesn't", "didn't",
        "won't", "wouldn't"
    }

    # Intensifiers can indicate exaggerated or emotional language.
    # This is relevant because fake or AI-generated reviews may sound
    # overly enthusiastic or overly polished.
    intensifier_words = {
        "very", "really", "extremely", "absolutely", "completely",
        "totally", "highly", "super", "so", "too", "incredibly"
    }

    # Certainty words can indicate overly confident or formulaic language.
    certainty_words = {
        "definitely", "certainly", "always", "clearly",
        "obviously", "undoubtedly", "surely", "guaranteed"
    }

    negation_count = sum(1 for word in word_texts if word in negation_words)
    intensifier_count = sum(1 for word in word_texts if word in intensifier_words)
    certainty_count = sum(1 for word in word_texts if word in certainty_words)

    # ---------------------------------------------------------------
    # 5. Sentiment features
    # ---------------------------------------------------------------
    # Sentiment is not pure stylometry, but it is useful for review data
    # because reviews are opinion-based texts.
    # Polarity ranges from negative to positive.
    # Subjectivity ranges from more factual to more opinion-based.
    try:
        blob = TextBlob(text)
        sentiment_polarity = blob.sentiment.polarity
        sentiment_subjectivity = blob.sentiment.subjectivity
    except:
        sentiment_polarity = 0
        sentiment_subjectivity = 0

    # ---------------------------------------------------------------
    # 6. Readability features
    # ---------------------------------------------------------------
    # Readability scores describe how easy or difficult the text is to read.
    # AI-generated text may sometimes be more polished and readable than
    # human-written reviews.

    # Flesch Reading Ease
    try:
        flesch_reading_ease = textstat.flesch_reading_ease(text)
    except:
        flesch_reading_ease = 0

    # Flesch-Kincaid Grade

    try:
        flesch_kincaid_grade = textstat.flesch_kincaid_grade(text)
    except:
        flesch_kincaid_grade = 0

    # Gunning Fog Index

    try:
        gunning_fog = textstat.gunning_fog(text)
    except:
        gunning_fog = 0

    syllable_count = count_syllables_in_text(text)

    # ---------------------------------------------------------------
    # 7. Repetition features
    # ---------------------------------------------------------------
    # These features capture repeated words or repeated punctuation.
    # They can indicate emphasis, informal writing, or formulaic patterns.
    repeated_punctuation_count = len(re.findall(r"([!?.,])\1+", text))

    repeated_word_count = sum(
        1 for i in range(1, len(word_texts))
        if word_texts[i] == word_texts[i - 1]
    )

    # ---------------------------------------------------------------
    # Store all features in a dictionary
    # ---------------------------------------------------------------
    # Ratios are used whenever possible because raw counts depend on
    # review length. For example, 5 commas mean something different in a
    # very short review than in a very long review.
    features = {
        # Length and structure
        "char_count": char_count,
        "word_count": word_count,
        "unique_word_count": unique_word_count,
        "sentence_count": sentence_count,
        "avg_word_length": np.mean(word_lengths) if word_lengths else 0,
        "std_word_length": np.std(word_lengths) if word_lengths else 0,
        "avg_sentence_length": np.mean(sentence_lengths) if sentence_lengths else 0,
        "std_sentence_length": np.std(sentence_lengths) if sentence_lengths else 0,

        # Lexical diversity
        "type_token_ratio": safe_divide(unique_word_count, word_count),

        # Punctuation ratios
        "punctuation_ratio": safe_divide(punctuation_count, char_count),
        "comma_ratio": safe_divide(comma_count, char_count),
        "period_ratio": safe_divide(period_count, char_count),
        "exclamation_ratio": safe_divide(exclamation_count, char_count),
        "question_ratio": safe_divide(question_count, char_count),
        "semicolon_ratio": safe_divide(semicolon_count, char_count),
        "colon_ratio": safe_divide(colon_count, char_count),
        "repeated_punctuation_ratio": safe_divide(repeated_punctuation_count, char_count),

        # Capitalization and numbers
        "uppercase_ratio": safe_divide(sum(1 for c in text if c.isupper()), char_count),
        "digit_ratio": safe_divide(sum(1 for c in text if c.isdigit()), char_count),

        # POS ratios
        "noun_ratio": safe_divide(noun_count, word_count),
        "verb_ratio": safe_divide(verb_count, word_count),
        "adjective_ratio": safe_divide(adj_count, word_count),
        "adverb_ratio": safe_divide(adv_count, word_count),
        "pronoun_ratio": safe_divide(pronoun_count, word_count),
        "conjunction_ratio": safe_divide(conjunction_count, word_count),

        # Function words and style markers
        "stopword_ratio": safe_divide(stopword_count, word_count),
        "negation_ratio": safe_divide(negation_count, word_count),
        "intensifier_ratio": safe_divide(intensifier_count, word_count),
        "certainty_ratio": safe_divide(certainty_count, word_count),

        # Readability
        "syllable_count": syllable_count,
        "flesch_reading_ease": flesch_reading_ease,
        "flesch_kincaid_grade": flesch_kincaid_grade,
        "gunning_fog": gunning_fog,

        # Sentiment
        "sentiment_polarity": sentiment_polarity,
        "sentiment_subjectivity": sentiment_subjectivity,

        # Repetition
        "repeated_word_count": repeated_word_count,
        "repeated_word_ratio": safe_divide(repeated_word_count, word_count)
    }

    return features


In [20]:
# -------------------------------------------------------------------
# Extract stylometric features for train, validation, and test data
# -------------------------------------------------------------------

print("\nExtracting train features...")
X_train = pd.DataFrame(
    train_df["text"].progress_apply(extract_stylometric_features).tolist()
)
y_train = train_df["ai_generated"]

print("\nExtracting test features...")
X_test = pd.DataFrame(
    test_df["text"].progress_apply(extract_stylometric_features).tolist()
)
y_test = test_df["ai_generated"]

print("\nNumber of stylometric features:", X_train.shape[1])
print("Train features:", X_train.shape)
print("Test features:", X_test.shape)

print("\nFeature preview:")
print(X_train.head())



Extracting train features...


100%|██████████| 7000/7000 [01:19<00:00, 87.63it/s] 



Extracting test features...


100%|██████████| 1500/1500 [00:16<00:00, 92.22it/s] 


Number of stylometric features: 37
Train features: (7000, 37)
Test features: (1500, 37)

Feature preview:
   char_count  word_count  unique_word_count  sentence_count  avg_word_length  std_word_length  avg_sentence_length  std_sentence_length  type_token_ratio  punctuation_ratio  comma_ratio  \
0         148          27                 25               2         4.370370         2.002742            13.500000             1.500000          0.925926           0.027027     0.013514   
1         205          39                 36               3         3.948718         1.568091            13.000000             2.449490          0.923077           0.043902     0.014634   
2         137          19                 15               3         3.473684         1.983310             6.333333             5.436502          0.789474           0.080292     0.007299   
3         141          28                 25               4         3.821429         1.513123             7.000000             1.870

In [21]:
# -------------------------------------------------------------------
# Save extracted stylometric features locally
# -------------------------------------------------------------------

import os
save_path = "Data"
os.makedirs(save_path, exist_ok=True)

X_train.to_csv(f"{save_path}/X_train_stylometric.csv", index=False)
X_test.to_csv(f"{save_path}/X_test_stylometric.csv", index=False)

y_train.to_csv(f"{save_path}/y_train.csv", index=False)
y_test.to_csv(f"{save_path}/y_test.csv", index=False)


In [22]:
# -------------------------------------------------------------------
# Load extracted stylometric features (local)
# -------------------------------------------------------------------

save_path = "Data"

X_train = pd.read_csv(f"{save_path}/X_train_stylometric.csv")
X_test = pd.read_csv(f"{save_path}/X_test_stylometric.csv")

y_train = pd.read_csv(f"{save_path}/y_train.csv").squeeze()
y_test = pd.read_csv(f"{save_path}/y_test.csv").squeeze()

print("Stylometric features loaded.")
print("Train features:", X_train.shape)
print("Test features:", X_test.shape)

Stylometric features loaded.
Train features: (7000, 37)
Test features: (1500, 37)


In [23]:
# -------------------------------------------------------------------
# Logistic Regression with stylometric features
# -------------------------------------------------------------------
# Goal:
# Train Logistic Regression on stylometric features.
# Tune C and penalty using 5-fold cross-validation on training data.
# Evaluate final model on the test set.
#
# Label coding:
# 1 = AI-generated
# 0 = human-written
# -------------------------------------------------------------------

from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__penalty": ["l1", "l2"]
}

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        solver="liblinear",
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
best_log_reg_model = grid.best_estimator_

print("\nBest C:", grid.best_params_["classifier__C"])
print("Best penalty:", grid.best_params_["classifier__penalty"])
print("Best CV F1:", round(grid.best_score_, 4))

best_C = grid.best_params_["classifier__C"]
best_penalty = grid.best_params_["classifier__penalty"]

Fitting 5 folds for each of 10 candidates, totalling 50 fits


c:\Users\behna\miniconda3\envs\fake-review\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\behna\miniconda3\envs\fake-review\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



Best C: 1
Best penalty: l1
Best CV F1: 0.8254


In [24]:
# -------------------------------------------------------------------
# Evaluation function
# -------------------------------------------------------------------
# This function reports the main classification metrics.
# Since label 1 = AI-generated, F1, precision, and recall refer to
# the AI-generated class by default.
# -------------------------------------------------------------------

def evaluate_model(model, X, y, model_name):
    y_pred = model.predict(X)

    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)

    print(f"\n=== {model_name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")

    if hasattr(model, "predict_proba"):
        y_proba_ai = model.predict_proba(X)[:, 1]
        auc = roc_auc_score(y, y_proba_ai)
        print(f"ROC-AUC:   {auc:.4f}")

    print("\nConfusion matrix:")
    print(confusion_matrix(y, y_pred, labels=[0, 1]))
    print("Label order: 0 = human-written, 1 = AI-generated")

    print("\nClassification report:")
    print(classification_report(
        y,
        y_pred,
        labels=[0, 1],
        target_names=["human-written", "AI-generated"]
    ))

In [25]:
# -------------------------------------------------------------------
# Evaluate Logistic Regression model
# -------------------------------------------------------------------
# Tune with 5-fold cross-validation
# ... grid.best_estimator_ is best_log_reg_model ...

# Report train and test performance
evaluate_model(
    best_log_reg_model,
    X_train,
    y_train,
    "Train Set"
)

evaluate_model(
    best_log_reg_model,
    X_test,
    y_test,
    "Test Set"
)


=== Train Set ===
Accuracy:  0.8141
F1-Score:  0.8263
Precision: 0.7756
Recall:    0.8840
ROC-AUC:   0.8826

Confusion matrix:
[[2605  895]
 [ 406 3094]]
Label order: 0 = human-written, 1 = AI-generated

Classification report:
               precision    recall  f1-score   support

human-written       0.87      0.74      0.80      3500
 AI-generated       0.78      0.88      0.83      3500

     accuracy                           0.81      7000
    macro avg       0.82      0.81      0.81      7000
 weighted avg       0.82      0.81      0.81      7000


=== Test Set ===
Accuracy:  0.7953
F1-Score:  0.8068
Precision: 0.7640
Recall:    0.8547
ROC-AUC:   0.8788

Confusion matrix:
[[552 198]
 [109 641]]
Label order: 0 = human-written, 1 = AI-generated

Classification report:
               precision    recall  f1-score   support

human-written       0.84      0.74      0.78       750
 AI-generated       0.76      0.85      0.81       750

     accuracy                           0.80     

In [26]:
# -------------------------------------------------------------------
# Leave-One-Category-Out Evaluation
# -------------------------------------------------------------------
# Goal:
# Test whether the stylometric Logistic Regression model generalizes
# to product categories that were not seen during training.
#
# For each category:
# - train on all other categories from the training set
# - test on the same held-out category from the test set
# -------------------------------------------------------------------
from sklearn.metrics import precision_recall_fscore_support

results_loc = []

categories = sorted(train_df["category"].unique())

for held_out in categories:
    print(f"\n=== Held-out category: {held_out} ===")

    # Train on all categories except the held-out one
    train_sub = train_df[train_df["category"] != held_out].copy()

    # Test only on the held-out category
    test_sub = test_df[test_df["category"] == held_out].copy()

    print("Train samples:", len(train_sub))
    print("Test samples:", len(test_sub))

    # Extract stylometric features
    X_tr = pd.DataFrame(
        train_sub["text"].progress_apply(extract_stylometric_features).tolist()
    )
    y_tr = train_sub["ai_generated"]

    X_te = pd.DataFrame(
        test_sub["text"].progress_apply(extract_stylometric_features).tolist()
    )
    y_te = test_sub["ai_generated"]

    # Logistic Regression model
    # Use the best C and penalty from cross-validation tuning
    m = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=best_C,
            penalty=best_penalty,
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=42
        ))
    ])

    # Train model
    m.fit(X_tr, y_tr)

    # Predict held-out category
    pred = m.predict(X_te)

    # Store results
    p_loc, r_loc, f_loc, s_loc = precision_recall_fscore_support(
        y_te, pred, labels=[0, 1], zero_division=0)

    results_loc.append({
        "category": held_out,
        "accuracy": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
        "f1": f1_score(y_te, pred, zero_division=0),
        "precision_human": p_loc[0],
        "recall_human": r_loc[0],
        "f1_human": f_loc[0],
        "precision_ai": p_loc[1],
        "recall_ai": r_loc[1],
        "f1_ai": f_loc[1],
        "samples": len(test_sub)
    })

# Convert results to dataframe
loc_df = pd.DataFrame(results_loc)

# Add average row
avg_row = pd.DataFrame([{
    "category": "AVERAGE",
    "accuracy": loc_df["accuracy"].mean(),
    "precision": loc_df["precision"].mean(),
    "recall": loc_df["recall"].mean(),
    "f1": loc_df["f1"].mean(),
    "precision_human": loc_df["precision_human"].mean(),
    "recall_human": loc_df["recall_human"].mean(),
    "f1_human": loc_df["f1_human"].mean(),
    "precision_ai": loc_df["precision_ai"].mean(),
    "recall_ai": loc_df["recall_ai"].mean(),
    "f1_ai": loc_df["f1_ai"].mean(),
    "samples": loc_df["samples"].sum()
}])

loc_df = pd.concat([loc_df, avg_row], ignore_index=True)

print("\n=== Leave-One-Category-Out Results ===")
print(loc_df)


=== Held-out category: Automotive ===
Train samples: 6629
Test samples: 79


100%|██████████| 79/79 [00:00<00:00, 96.60it/s] 
c:\Users\behna\miniconda3\envs\fake-review\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\behna\miniconda3\envs\fake-review\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



=== Held-out category: Beauty_and_Personal_Care ===
Train samples: 6507
Test samples: 101


100%|██████████| 101/101 [00:01<00:00, 93.40it/s]
c:\Users\behna\miniconda3\envs\fake-review\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\behna\miniconda3\envs\fake-review\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



=== Held-out category: Books ===
Train samples: 6565
Test samples: 78


...


=== Held-out category: Kindle_Store ===
Train samples: 6640
Test samples: 86


100%|██████████| 86/86 [00:01<00:00, 73.50it/s]
c:\Users\behna\miniconda3\envs\fake-review\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\behna\miniconda3\envs\fake-review\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



=== Held-out category: Tools_and_Home_Improvement ===
Train samples: 6427
Test samples: 110


100%|██████████| 110/110 [00:01<00:00, 99.17it/s] 
c:\Users\behna\miniconda3\envs\fake-review\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\behna\miniconda3\envs\fake-review\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



=== Leave-One-Category-Out Results ===
                       category  accuracy  accuracy_human  accuracy_ai  precision    recall        f1  precision_human  recall_human  f1_human  precision_ai  recall_ai     f1_ai  samples
0                    Automotive  0.810127        0.816327     0.800000   0.727273  0.800000  0.761905         0.869565      0.816327  0.842105      0.727273   0.800000  0.761905       79
1      Beauty_and_Personal_Care  0.752475        0.735849     0.770833   0.725490  0.770833  0.747475         0.780000      0.735849  0.757282      0.725490   0.770833  0.747475      101
2                         Books  0.846154        0.718750     0.934783   0.826923  0.934783  0.877551         0.884615      0.718750  0.793103      0.826923   0.934783  0.877551       78
3   Cell_Phones_and_Accessories  0.769231        0.784615     0.743590   0.674419  0.743590  0.707317         0.836066      0.784615  0.809524      0.674419   0.743590  0.707317      104
4    Clothing_Shoes_and_J

In [27]:
# -------------------------------------------------------------------
# Logistic Regression coefficient table
# -------------------------------------------------------------------
# Positive coefficients indicate features associated with AI-generated reviews.
# Negative coefficients indicate features associated with human-written reviews.
# -------------------------------------------------------------------

coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": best_log_reg_model.named_steps["classifier"].coef_[0]
})

# Add absolute coefficient size for sorting
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

# Add direction of association
coef_df["associated_class"] = coef_df["coefficient"].apply(
    lambda x: "AI-generated" if x > 0 else "human-written"
)

# Sort by largest absolute coefficient
coef_df_sorted = coef_df.sort_values("abs_coefficient", ascending=False)

# Show top 15 features
print("Top 15 stylometric features by absolute Logistic Regression coefficient:")
print(coef_df_sorted.head(15))

Top 15 stylometric features by absolute Logistic Regression coefficient:
                       feature  coefficient  abs_coefficient associated_class
1                   word_count   -23.365055        23.365055    human-written
2            unique_word_count    19.385517        19.385517     AI-generated
3               sentence_count    -5.022817         5.022817    human-written
16  repeated_punctuation_ratio    -3.774252         3.774252    human-written
8             type_token_ratio    -1.279154         1.279154    human-written
6          avg_sentence_length    -1.274624         1.274624    human-written
17             uppercase_ratio    -1.051762         1.051762    human-written
9            punctuation_ratio     0.976775         0.976775     AI-generated
21             adjective_ratio    -0.918356         0.918356    human-written
15                 colon_ratio    -0.875579         0.875579    human-written
20                  verb_ratio    -0.808435         0.808435    human

In [28]:
# -------------------------------------------------------------------
# Compare stylometric feature values between the two classes
# -------------------------------------------------------------------
# Goal:
# We want to identify which stylometric features differ most between
# human-written reviews and AI-generated reviews.
#
# Label coding:
# 0 = human-written
# 1 = AI-generated
# -------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

# Split the training data by class.
# We use the training set here because it is the data used to fit the model.
human_data = X_train[y_train == 0]
ai_data = X_train[y_train == 1]

# Calculate the average value of each feature for each class.
# This shows whether a feature tends to be higher for human-written
# or AI-generated reviews.
human_mean = human_data.mean()
ai_mean = ai_data.mean()

# Calculate the standard deviation of each feature in both classes.
# This is needed because features are measured on different scales.
human_std = human_data.std()
ai_std = ai_data.std()

# Raw difference between class means.
# Positive values mean the feature is higher on average in AI-generated reviews.
# Negative values mean the feature is higher on average in human-written reviews.
difference_ai_minus_human = ai_mean - human_mean

# Absolute raw difference.
# This shows the size of the raw difference, ignoring the direction.
abs_difference = difference_ai_minus_human.abs()

# Pooled standard deviation.
# This represents the typical variation of each feature across both classes.
# It is used to standardize the raw mean difference.
pooled_std = np.sqrt((human_std ** 2 + ai_std ** 2) / 2)

# Standardized mean difference.
# This makes features comparable even if they are measured on different scales.
# For example, word_count and punctuation_ratio cannot be compared fairly
# using raw differences only, because their numerical ranges are very different.
standardized_difference = difference_ai_minus_human / pooled_std

# Absolute standardized difference.
# This shows how strongly a feature differs between the two classes,
# regardless of whether it is higher for AI-generated or human-written reviews.
abs_standardized_difference = standardized_difference.abs()

# Combine all results into one table.
feature_difference_df = pd.DataFrame({
    "feature": X_train.columns,
    "human_mean": human_mean.values,
    "ai_mean": ai_mean.values,
    "difference_ai_minus_human": difference_ai_minus_human.values,
    "abs_difference": abs_difference.values,
    "standardized_difference": standardized_difference.values,
    "abs_standardized_difference": abs_standardized_difference.values
})

# Sort by absolute standardized difference.
# The features at the top are those with the strongest class differences
# after accounting for the different scales of the features.
feature_difference_df = feature_difference_df.sort_values(
    "abs_standardized_difference",
    ascending=False
)

display(feature_difference_df.head(20))

,feature,human_mean,ai_mean,difference_ai_minus_human,abs_difference,standardized_difference,abs_standardized_difference
8,type_token_ratio,0.868877,0.930139,0.061261,0.061261,0.614650,0.614650
3,sentence_count,3.605143,2.338571,-1.266571,1.266571,-0.456773,0.456773
1,word_count,41.617143,22.255143,-19.362000,19.362000,-0.456627,0.456627
2,unique_word_count,30.614571,20.187714,-10.426857,10.426857,-0.436859,0.436859
0,char_count,222.835429,122.656857,-100.178571,100.178571,-0.435665,0.435665
29,syllable_count,56.872571,31.607429,-25.265143,25.265143,-0.424903,0.424903
5,std_word_length,1.967276,2.163706,0.196430,0.196430,0.393405,0.393405
11,period_ratio,0.015972,0.021300,0.005328,0.005328,0.384883,0.384883
16,repeated_punctuation_ratio,0.000891,0.000001,-0.000890,0.000890,-0.293356,0.293356
4,avg_word_length,4.251880,4.432523,0.180643,0.180643,0.291668,0.291668
